# Notebook 07 — Report Figures
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Export all final publication-quality maps and charts for the technical summary report.

**Run this notebook last — after all analysis notebooks (01–06) are complete.**

---
## Figures produced
| Figure | File | Description |
|--------|------|-------------|
| Fig 1 | `fig01_borehole_network.png` | All 706 boreholes by status and GPS quality |
| Fig 2 | `fig02_wasi_choropleth.png` | Water Access Stress Index — ward choropleth |
| Fig 3 | `fig03_wasi_components.png` | WASI component breakdown panel |
| Fig 4 | `fig04_hotspot_map.png` | Gi* hotspot/coldspot classification |
| Fig 5 | `fig05_coverage_gap.png` | 2km service area coverage and gap |
| Fig 6 | `fig06_site_rankings.png` | Top 20 IoT pilot candidates (interim 45-pt) |
| Fig 7 | `fig07_ward_summary_table.png` | Full ward summary table (all metrics) |
| Tab 1 | `table01_wasi_ward.csv` | WASI ward table for report |
| Tab 2 | `table02_coverage_gap.csv` | Coverage gap ward table |
| Tab 3 | `table03_interim_rankings.csv` | Interim borehole rankings |

All outputs saved to `outputs/maps/` and `outputs/report/`.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install geopandas matplotlib seaborn plotly openpyxl -q

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'
OUT   = DRIVE + 'outputs/'
MAPS  = OUT + 'maps/'
REPT  = OUT + 'report/'

import os
os.makedirs(MAPS, exist_ok=True)
os.makedirs(REPT, exist_ok=True)

WGS84 = 'EPSG:4326'

# ── Style settings ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':     'DejaVu Sans',
    'font.size':       10,
    'axes.titlesize':  11,
    'axes.labelsize':  9,
    'figure.dpi':      150,
    'savefig.dpi':     300,
    'savefig.bbox':    'tight',
})

BRAND = {
    'blue':        '#0B5394',
    'light_blue':  '#2E75B6',
    'red':         '#C00000',
    'orange':      '#ED7D31',
    'green':       '#70AD47',
    'yellow':      '#FFC000',
    'grey':        '#808080',
    'light_grey':  '#D9D9D9',
}

def add_north_arrow(ax, x=0.95, y=0.10):
    ax.annotate('N', xy=(x, y), xytext=(x, y - 0.05),
                xycoords='axes fraction',
                fontsize=12, ha='center', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

def add_caption(ax, text):
    ax.text(0.5, -0.08, text, transform=ax.transAxes,
            ha='center', va='top', fontsize=7.5, style='italic', color='#555555')

print('Setup complete — run after all analysis notebooks')

In [ ]:
# ── Load all analysis outputs ─────────────────────────────────────────────────

# Borehole master dataset
df_bh = pd.read_excel(
    DRIVE + 'Kitui_Boreholes_Master_Dataset.xlsx',
    sheet_name='B_Spatial_Analysis', header=2
)
gdf_bh = gpd.GeoDataFrame(
    df_bh[df_bh['GPS_Available'] == True],
    geometry=[Point(xy) for xy in zip(
        df_bh.loc[df_bh['GPS_Available'] == True, 'Longitude'],
        df_bh.loc[df_bh['GPS_Available'] == True, 'Latitude']
    )],
    crs=WGS84
)

# Ward boundaries
wards = gpd.read_file(DRIVE + 'boundaries/kitui_wards.shp').to_crs(WGS84)

# WASI outputs
wasi_table  = pd.read_csv(OUT + 'kitui_wasi_ward_table.csv')
wasi_wards  = gpd.read_file(OUT + 'kitui_wasi_ward.geojson')

# Hotspot outputs
hotspot_wards = gpd.read_file(OUT + 'kitui_hotspot_ward.geojson')

# Coverage gap (from Notebook 04)
try:
    coverage_table = pd.read_csv(OUT + 'kitui_coverage_gap_ward_table.csv')
    coverage_gap   = gpd.read_file(OUT + 'kitui_coverage_gap.geojson')
    has_coverage   = True
except FileNotFoundError:
    has_coverage = False
    print('Coverage gap outputs not found — run Notebook 04 first')

# Rankings (from Notebook 05)
try:
    rankings = pd.read_excel(OUT + 'kitui_borehole_interim_rankings.xlsx')
    has_rankings = True
except FileNotFoundError:
    has_rankings = False
    print('Rankings output not found — run Notebook 05 first')

print('Data loaded successfully')
print(f'Boreholes (GPS): {len(gdf_bh)} | WASI wards: {len(wasi_wards)} | Hotspot wards: {len(hotspot_wards)}')

In [ ]:
# ── Figure 1: Borehole network overview ───────────────────────────────────────

STATUS_COLOURS = {
    'Functional':                       BRAND['light_blue'],
    'Functional - Needs Rehabilitation': BRAND['yellow'],
    'Functional - At Risk':              BRAND['orange'],
    'Partially Functional':              '#FF9900',
    'Non-Functional':                    BRAND['red'],
    'Non-Functional - Not Equipped':     '#FF6666',
    'Non-Functional - Abandoned':        '#AA0000',
    'Under Construction':                BRAND['green'],
    'Unknown':                           BRAND['grey'],
}

fig, axes = plt.subplots(1, 2, figsize=(18, 12))

# Panel A: Status
ax = axes[0]
wards.plot(ax=ax, color='#F5F5F5', edgecolor='#CCCCCC', linewidth=0.5)
for status, colour in STATUS_COLOURS.items():
    subset = gdf_bh[gdf_bh['Functionality_Status'] == status]
    if len(subset) > 0:
        subset.plot(ax=ax, color=colour, markersize=4, alpha=0.8, label=status)
ax.legend(fontsize=7, loc='lower left', title='Status', title_fontsize=8)
ax.set_title(f'Borehole Network — Functionality Status\n(n={len(gdf_bh)} GPS-available, {len(df_bh)} total)', fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, f'Source: Kitui County inventory + mWater | GPS available: {len(gdf_bh)}/{len(df_bh)}')

# Panel B: GPS quality
ax = axes[1]
GPS_COLOURS = {
    'Verified':                          BRAND['light_blue'],
    'Needs Review - Shared Coordinates': BRAND['yellow'],
    'Low Confidence Match':              BRAND['orange'],
}
wards.plot(ax=ax, color='#F5F5F5', edgecolor='#CCCCCC', linewidth=0.5)
for quality, colour in GPS_COLOURS.items():
    subset = gdf_bh[gdf_bh['GPS_Quality'] == quality]
    if len(subset) > 0:
        subset.plot(ax=ax, color=colour, markersize=4, alpha=0.8, label=f'{quality} (n={len(subset)})')
ax.legend(fontsize=7, loc='lower left', title='GPS Quality', title_fontsize=8)
ax.set_title(f'Borehole Network — GPS Quality\n({(df_bh["GPS_Quality"]=="No GPS").sum()} boreholes have no GPS and are not shown)', fontsize=11)
ax.set_xlabel('Longitude')
add_north_arrow(ax)
add_caption(ax, 'GPS quality affects which records can be used in spatial analysis')

plt.suptitle('Kitui County Borehole Network — 706 Boreholes, 40 Wards',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
path = MAPS + 'fig01_borehole_network.png'
plt.savefig(path, dpi=300)
plt.show()
print(f'Fig 1 saved: {path}')

In [ ]:
# ── Figure 2: WASI ward choropleth ────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 14))

wasi_wards.plot(
    column='WASI_mean', cmap='RdYlGn_r', linewidth=0.6, edgecolor='white',
    legend=True, vmin=0, vmax=1,
    legend_kwds={'label': 'Water Access Stress Index (0=low, 1=high)',
                 'orientation': 'vertical', 'shrink': 0.7},
    ax=ax
)

# Label top 10 stress wards
top10 = wasi_wards.nlargest(10, 'WASI_mean')
for _, row in top10.iterrows():
    c = row.geometry.centroid
    ax.annotate(row['Ward'], xy=(c.x, c.y), fontsize=6.5, ha='center', va='center',
                fontweight='bold', color='black',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'))

# Borehole overlay (verified functional only)
func_verified = gdf_bh[
    (gdf_bh['Is_Functional'] == True) & (gdf_bh['GPS_Quality'] == 'Verified')
]
func_verified.plot(ax=ax, color=BRAND['blue'], markersize=3, alpha=0.5,
                   label=f'Functional BH, GPS-verified (n={len(func_verified)})')

ax.legend(loc='lower left', fontsize=8)
ax.set_title('Water Access Stress Index — Kitui County\nWard-level composite: distance (30%) + rainfall (25%) + NDVI (15%) + population (20%) + slope (10%)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'WASI components normalised 0-1 and weighted. Source: MODIS/CHIRPS/SRTM/WorldPop via GEE.')

plt.tight_layout()
path = MAPS + 'fig02_wasi_choropleth.png'
plt.savefig(path, dpi=300)
plt.show()
print(f'Fig 2 saved: {path}')

In [ ]:
# ── Figure 3: WASI component bar chart ────────────────────────────────────────

wasi_sorted = wasi_table.sort_values('WASI_mean', ascending=True)

comp_cols   = ['C1_Distance', 'C2_Rainfall', 'C3_NDVI', 'C4_Population', 'C5_Slope']
comp_labels = ['Distance (30%)', 'Rainfall (25%)', 'NDVI (15%)', 'Population (20%)', 'Slope (10%)']
comp_weights = [0.30, 0.25, 0.15, 0.20, 0.10]
comp_colors  = [BRAND['red'], BRAND['orange'], BRAND['green'], BRAND['light_blue'], BRAND['yellow']]

# Weighted contribution per component per ward
fig, ax = plt.subplots(figsize=(14, 10))

y_pos = np.arange(len(wasi_sorted))
cumulative = np.zeros(len(wasi_sorted))

for col, label, w, colour in zip(comp_cols, comp_labels, comp_weights, comp_colors):
    vals = wasi_sorted[col].fillna(0).values * w
    ax.barh(y_pos, vals, left=cumulative, height=0.7,
            color=colour, alpha=0.85, label=label)
    cumulative += vals

# WASI total line
ax.scatter(wasi_sorted['WASI_mean'].values, y_pos, color='black',
           s=15, zorder=5, label='WASI composite')

ax.set_yticks(y_pos)
ax.set_yticklabels(wasi_sorted['Ward'], fontsize=7)
ax.set_xlabel('Weighted stress score (0 = no stress, 1 = maximum stress)')
ax.set_title('WASI Component Breakdown by Ward — Kitui County',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.axvline(0.55, color='red', linestyle='--', alpha=0.4, label='High stress threshold')
ax.set_xlim(0, 1.0)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
path = MAPS + 'fig03_wasi_components.png'
plt.savefig(path, dpi=300)
plt.show()
print(f'Fig 3 saved: {path}')

In [ ]:
# ── Figure 4: Hotspot map (from Notebook 03 output) ──────────────────────────
import shutil

src = OUT + 'kitui_hotspot_map.png'
dst = MAPS + 'fig04_hotspot_map.png'
if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'Fig 4 copied: {dst}')
else:
    print(f'Hotspot map not found at {src} — run Notebook 03 first')

# Regenerate a clean version for the report
HOTSPOT_COLOURS = {
    'Hotspot (99%)':  '#C00000', 'Hotspot (95%)':  '#FF6666', 'Hotspot (90%)':  '#FFAAAA',
    'Not significant':'#E0E0E0',
    'Coldspot (90%)': '#AACCEE', 'Coldspot (95%)': '#2E75B6', 'Coldspot (99%)': '#0B5394',
}

fig, ax = plt.subplots(figsize=(12, 14))
for cls, colour in HOTSPOT_COLOURS.items():
    subset = hotspot_wards[hotspot_wards['Hotspot_Class'] == cls]
    if len(subset) > 0:
        subset.plot(ax=ax, color=colour, linewidth=0.6, edgecolor='white')

# Ward boundary outline
hotspot_wards.boundary.plot(ax=ax, color='#AAAAAA', linewidth=0.4)

# Label hotspot wards
for _, row in hotspot_wards[hotspot_wards['Hotspot_Class'].str.startswith('Hotspot', na=False)].iterrows():
    c = row.geometry.centroid
    ax.annotate(row['Ward'], xy=(c.x, c.y), fontsize=6.5, ha='center', va='center',
                fontweight='bold', color='white')

legend_patches = [
    mpatches.Patch(color='#C00000', label='Hotspot p<0.01'),
    mpatches.Patch(color='#FF6666', label='Hotspot p<0.05'),
    mpatches.Patch(color='#E0E0E0', label='Not significant'),
    mpatches.Patch(color='#2E75B6', label='Coldspot p<0.05'),
    mpatches.Patch(color='#0B5394', label='Coldspot p<0.01'),
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=9, title='Gi* Cluster Type')
ax.set_title('Water Stress Hotspot Analysis — Kitui County\nGetis-Ord Gi* | Queen contiguity weights | 999 permutations',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'Hotspot = ward with significantly elevated stress surrounded by high-stress neighbours. Source: WASI (Notebook 02).')

plt.tight_layout()
plt.savefig(MAPS + 'fig04_hotspot_map.png', dpi=300)
plt.show()
print(f'Fig 4 saved: {MAPS}fig04_hotspot_map.png')

In [ ]:
# ── Figure 5: Coverage gap map ────────────────────────────────────────────────

if has_coverage:
    fig, axes = plt.subplots(1, 2, figsize=(18, 12))

    # Panel A: coverage map
    ax = axes[0]
    wards.plot(ax=ax, color='#FFE0B2', edgecolor='#CCCCCC', linewidth=0.5)
    coverage_gap.plot(ax=ax, color=BRAND['red'], alpha=0.4, label='Population gap (>2km from BH)')
    func_verified.plot(ax=ax, color=BRAND['blue'], markersize=4, alpha=0.6,
                       label='Functional BH (GPS verified)')
    ax.legend(fontsize=8, loc='lower left')
    ax.set_title('Borehole Service Coverage (2km walking threshold)\nRed = population outside coverage', fontsize=11)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    add_north_arrow(ax)

    # Panel B: gap % choropleth
    ax = axes[1]
    coverage_wards = wards.merge(coverage_table, on='Ward', how='left')
    coverage_wards.plot(
        column='Pct_Pop_Gap', cmap='Reds', linewidth=0.5, edgecolor='white',
        legend=True, vmin=0, vmax=100,
        legend_kwds={'label': '% population outside 2km coverage', 'orientation': 'vertical'},
        ax=ax
    )
    ax.set_title('Population Gap by Ward (%)\n% of ward population outside 2km of any functional borehole', fontsize=11)
    ax.set_xlabel('Longitude')
    add_north_arrow(ax)

    plt.suptitle('Borehole Coverage Gap Analysis — Kitui County (2km walking threshold)',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    path = MAPS + 'fig05_coverage_gap.png'
    plt.savefig(path, dpi=300)
    plt.show()
    print(f'Fig 5 saved: {path}')
else:
    print('Skipping Fig 5 — coverage gap outputs not available (run Notebook 04)')

In [ ]:
# ── Figure 6: Top 20 interim IoT site rankings ─────────────────────────────────

if has_rankings:
    top20 = rankings.head(20).copy()
    top20['Label'] = top20['Borehole_Name'].str[:25] + '\n(' + top20['Ward'] + ')'

    fig, ax = plt.subplots(figsize=(14, 10))

    colours = [
        BRAND['light_blue'] if mgmt in ['Kitwasco', 'Kimwasco'] else
        BRAND['green'] if mgmt in ['Project Maji', 'FundiFix'] else
        BRAND['yellow']
        for mgmt in top20['Management_Type']
    ]

    bars = ax.barh(range(len(top20)), top20['Score_45'], color=colours, alpha=0.85)
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(top20['Label'], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Score (out of 45 available pts — 55 field assessment pts pending)')
    ax.set_title('Top 20 IoT Pilot Candidates — Interim Ranking (45/100 pts)\n'
                 'CAUTION: Final ranking requires 55 field-assessed points',
                 fontsize=11, fontweight='bold')
    ax.set_xlim(0, 47)
    ax.axvline(35, color='orange', linestyle='--', alpha=0.5, label='35/45 — strong interim candidate')
    ax.axvline(45, color='green',  linestyle='--', alpha=0.5, label='45/45 — maximum possible')

    # Score labels on bars
    for bar, score in zip(bars, top20['Score_45']):
        ax.text(score + 0.3, bar.get_y() + bar.get_height()/2,
                f'{score}', va='center', fontsize=8)

    legend_patches = [
        mpatches.Patch(color=BRAND['light_blue'], label='WSP (Kitwasco/Kimwasco)'),
        mpatches.Patch(color=BRAND['green'],      label='Professional NGO'),
        mpatches.Patch(color=BRAND['yellow'],     label='Other management'),
    ]
    ax.legend(handles=legend_patches + [
        plt.Line2D([0], [0], color='orange', linestyle='--', label='35/45 threshold'),
    ], fontsize=8, loc='lower right')

    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = MAPS + 'fig06_site_rankings.png'
    plt.savefig(path, dpi=300)
    plt.show()
    print(f'Fig 6 saved: {path}')
else:
    print('Skipping Fig 6 — rankings output not available (run Notebook 05)')

In [ ]:
# ── Export report tables ──────────────────────────────────────────────────────

# Table 1: WASI ward summary
t1 = wasi_table.sort_values('WASI_mean', ascending=False).copy()
t1.columns = [c.replace('_', ' ') for c in t1.columns]
t1.to_csv(REPT + 'table01_wasi_ward.csv', index=False)
print(f'Table 1 saved: {REPT}table01_wasi_ward.csv')

# Table 2: Coverage gap
if has_coverage:
    coverage_table.to_csv(REPT + 'table02_coverage_gap.csv', index=False)
    print(f'Table 2 saved: {REPT}table02_coverage_gap.csv')

# Table 3: Interim rankings
if has_rankings:
    rankings.to_csv(REPT + 'table03_interim_rankings.csv', index=False)
    print(f'Table 3 saved: {REPT}table03_interim_rankings.csv')

# List all outputs
print()
print('── REPORT FIGURES COMPLETE ──────────────────────────────────────')
print('Maps saved to:   ', MAPS)
print('Tables saved to: ', REPT)
print()
for f in sorted(os.listdir(MAPS)):
    print(f'  {MAPS}{f}')
for f in sorted(os.listdir(REPT)):
    print(f'  {REPT}{f}')